**Работа с API**

1.1 Получение информации обо всех книгах и сохранение в pd.DataFrame

In [3]:
books = pd.DataFrame(fetch_all("books"))
print(f"книг получено: {len(books)}")
print(f"столбцы: {list(books.columns)}")
books[["name", "authors", "numberOfPages", "publisher", "released"]]

книг получено: 12
столбцы: ['url', 'name', 'isbn', 'authors', 'numberOfPages', 'publisher', 'country', 'mediaType', 'released', 'characters', 'povCharacters']


,name,authors,numberOfPages,publisher,released
0,A Game of Thrones,[George R. R. Martin],694,Bantam Books,1996-08-01T00:00:00
1,A Clash of Kings,[George R. R. Martin],768,Bantam Books,1999-02-02T00:00:00
2,A Storm of Swords,[George R. R. Martin],992,Bantam Books,2000-10-31T00:00:00
3,The Hedge Knight,[George R. R. Martin],164,Dabel Brothers Publishing,2005-03-09T00:00:00
4,A Feast for Crows,[George R. R. Martin],784,Bantam Books,2005-11-08T00:00:00
5,The Sworn Sword,[George R. R. Martin],152,Marvel,2008-06-18T00:00:00
6,The Mystery Knight,[George R. R. Martin],416,Tor Fantasy,2011-03-29T00:00:00
7,A Dance with Dragons,[George R. R. Martin],1040,Bantam Books,2011-07-12T00:00:00
8,The Princess and the Queen,[George R. R. Martin],784,Tor Books,2013-12-03T00:00:00
9,The Rogue Prince,[George R. R. Martin],832,Bantam Books,2014-06-17T00:00:00


1.2 Получение информации обо всех домах Вестероса (Houses) и сохранение в pd.DataFrame

In [4]:
houses = pd.DataFrame(fetch_all("houses"))
print(f"домов получено: {len(houses)}")
print(f"столбцы: {list(houses.columns)}")
houses[["name", "region", "words", "coatOfArms", "founded"]].head(10)

домов получено: 444
столбцы: ['url', 'name', 'region', 'coatOfArms', 'words', 'titles', 'seats', 'currentLord', 'heir', 'overlord', 'founded', 'founder', 'diedOut', 'ancestralWeapons', 'cadetBranches', 'swornMembers']


,name,region,words,coatOfArms,founded
0,House Algood,The Westerlands,,"A golden wreath, on a blue field with a gold b...",
1,House Allyrion of Godsgrace,Dorne,No Foe May Pass,"Gyronny Gules and Sable, a hand couped Or",
2,House Amber,The North,,,
3,House Ambrose,The Reach,Never Resting,"Or, semy of ants gules",
4,House Appleton of Appleton,The Reach,,"Or, an apple tree eradicated proper fructed gu...",
5,House Arryn of Gulltown,The Vale,,,
6,House Arryn of the Eyrie,The Vale,As High as Honor,A sky-blue falcon soaring against a white moon...,Coming of the Andals
7,House Ashford of Ashford,The Reach,Our Sun Shines Bright,"Tenny, a sun in splendour beneath a chevron in...",
8,House Ashwood,The North,,,
9,House Baelish of Harrenhal,The Riverlands,,"A field of silver mockingbirds, on a green fie...",299 AC


1.3 Получение информации обо всех домах Вестероса, у которых есть девиз, и сохранение в pd.DataFrame

In [11]:
houses = pd.DataFrame(fetch_all("houses"))

houses_with_motto = pd.DataFrame(fetch_all("houses", hasTitles="true"))

print(f"hasTitles=true  (Есть девиз):  {len(houses_with_motto):>3} домов")
print(f"всего домов в Вестеросе:            {len(houses):>3}")


hasTitles=true  (Есть девиз):  165 домов
всего домов в Вестеросе:            444


**Работа с БД**

2.1 Установка библиотеки psycopg2

In [12]:
pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.8 MB/s  0:00:00


2.2 Подключение к БД

In [14]:
import time

import psycopg2

DB_PARAMS = {
    "host": "hh-pgsql-public.ebi.ac.uk",
    "port": 5432,
    "dbname": "pfmegrnargs",
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "connect_timeout": 20,
}


def connect(attempts: int = 5) -> psycopg2.extensions.connection:

    for attempt in range(1, attempts + 1):
        try:
            return psycopg2.connect(**DB_PARAMS)
        except psycopg2.OperationalError as error:
            if attempt == attempts:
                raise
            print(f"попытка {attempt}/{attempts} не удалась: "
                  f"{str(error).strip().splitlines()[0]}")
            time.sleep(5 * attempt)
connection = connect()
print("подключение установлено")
print("сервер:", connection.get_dsn_parameters()["host"])
print("база:  ", connection.get_dsn_parameters()["dbname"])

подключение установлено
сервер: hh-pgsql-public.ebi.ac.uk
база:   pfmegrnargs


2.2 Получение 10 строк из таблицы rnc_database

In [15]:
cursor = connection.cursor()
cursor.execute("SELECT * FROM rnc_database LIMIT 10")
rows = cursor.fetchall()

print(f"Получено строк: {len(rows)}")
print(f"значений в строке: {len(rows[0])}")
rows[0][:6]

Получено строк: 10
значений в строке: 20


(5, datetime.datetime(2017, 5, 17, 0, 0), 'RNACEN', 'VEGA', 98, 'VEGA')

2.3 Сохранение результата в DataFrame

In [16]:
column_names = [column[0] for column in cursor.description]

databases = pd.DataFrame(rows, columns=column_names)
print(f"строк: {databases.shape[0]}, столбцов: {databases.shape[1]}")
databases[["id", "display_name", "num_sequences", "alive"]]

строк: 10, столбцов: 20


,id,display_name,num_sequences,alive
0,5,VEGA,0,N
1,26,GENCODE,47677,N
2,25,Ensembl,3373550,Y
3,1,ENA,12086180,Y
4,59,circAtlas,2800521,Y
5,31,Ensembl Plants,167931,Y
6,32,LncBase,1352,Y
7,7,lncRNAdb,62,N
8,57,CIRCpedia,2530830,Y
9,46,piRBase,480870,Y


2.4 Получение значений столбцов display_name, num_sequences, num_organisms, , url для 10 строк из таблицы rnc_database

In [17]:
cursor.execute("""
    SELECT display_name, num_sequences, num_organisms, url
    FROM rnc_database
    LIMIT 10
""")
selected = pd.DataFrame(cursor.fetchall(),
                        columns=[column[0] for column in cursor.description])
selected

,display_name,num_sequences,num_organisms,url
0,VEGA,0,0,http://vega.sanger.ac.uk/
1,GENCODE,47677,2,http://gencodegenes.org/
2,Ensembl,3373550,267,http://ensembl.org/
3,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
4,circAtlas,2800521,10,https://ngdc.cncb.ac.cn/circatlas/tutorial1.php
5,Ensembl Plants,167931,69,https://plants.ensembl.org/
6,LncBase,1352,2,http://www.microrna.gr/LncBase
7,lncRNAdb,62,10,http://lncrnadb.org/
8,CIRCpedia,2530830,20,https://bits.fudan.edu.cn/circpediav3
9,piRBase,480870,26,http://www.regulatoryrna.org/database/piRNA/
